# RAG Fusion

In [ ]:
!pip -q install langchain huggingftiktace_hub oken pypdf
!pip -q install google-generativeai chromadb
!pip -q install sentence_transformers

In [ ]:
!pip install -U langchain-community

### Download the Data & Utils

In [2]:
import textwrap
def wrap_text(text, width=90): #preserve_newlines
    # Split the input text into lines based on newline characters
    lines = text.split('\n')

    # Wrap each line individually
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text


In [3]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


In [ ]:
%pip install --upgrade --quiet  langchain-google-genai

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

result = llm.invoke("What is RAG Fusion?")
print(result.content)

**RAG Fusion** is an advanced technique in Retrieval Augmented Generation (RAG) that aims to significantly improve the quality and comprehensiveness of the retrieved context provided to a Large Language Model (LLM).

The core idea behind RAG Fusion is to overcome the limitations of a single, static retrieval query by **generating multiple, diverse perspectives of the original user query** and then **fusing the results** from these multiple retrieval attempts.

Here's a breakdown of how it works and why it's powerful:

### The Problem RAG Fusion Solves

Traditional RAG often relies on a single query (the user's original question) to retrieve documents. This can be problematic because:
1.  **Query Ambiguity:** The user's query might be phrased in a way that isn't optimal for the underlying search index.
2.  **Missing Nuances:** A single query might not capture all the different facets or implied meanings of a complex request.
3.  **Suboptimal Retrieval:** If the initial retrieval fails t

## Google

## Imports

In [7]:
import langchain
print(langchain.__version__)

1.3.11


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import langchain


## Load in Docs

In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\2075362422.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [10]:
data_path=(r"F:\Advance RAG\Data\Multiple_text_data")

In [ ]:
!pip install unstructured

In [ ]:

%%time
loader = DirectoryLoader(data_path, glob="*.txt", show_progress=True)
docs = loader.load()

In [14]:
len(docs)

700

In [15]:
docs = docs[:50]
len(docs)

50

In [16]:
docs[0]

Document(metadata={'source': 'F:\\Advance RAG\\Data\\Multiple_text_data\\101_Best_Things_To_Do_In_Singapore.txt'}, page_content='Last updated March 26, 2026: Creative hubs in unexpected buildings are in, as are new nature experiences and immersive multimedia attractions. Specifically, there’s lots of buzz surrounding Mandai Wildlife Reserve and its fresh openings, while more independent arts spaces as well as thrift stores are having a moment.\n\nHow we choose the best things to do in Singapore:\n\nThe Time Out Singapore team is constantly on the lookout for the coolest new activities, great food, and the latest happenings at our cultural institutions – because we refuse to believe that being a small island city means being boring.\n\nFrom world-famous landmarks like Marina Bay Sands and Gardens by the Bay to island escapes in Sentosa, Singapore is packed with headline attractions, and all of these are celebrated for good reason. But scratch beneath the surface and there’s far more to 

In [17]:
from pprint import pprint

pprint(docs[0].metadata)
print("\nContent:\n")
print(docs[0].page_content)

{'source': 'F:\\Advance '
           'RAG\\Data\\Multiple_text_data\\101_Best_Things_To_Do_In_Singapore.txt'}

Content:

Last updated March 26, 2026: Creative hubs in unexpected buildings are in, as are new nature experiences and immersive multimedia attractions. Specifically, there’s lots of buzz surrounding Mandai Wildlife Reserve and its fresh openings, while more independent arts spaces as well as thrift stores are having a moment.

How we choose the best things to do in Singapore:

The Time Out Singapore team is constantly on the lookout for the coolest new activities, great food, and the latest happenings at our cultural institutions – because we refuse to believe that being a small island city means being boring.

From world-famous landmarks like Marina Bay Sands and Gardens by the Bay to island escapes in Sentosa, Singapore is packed with headline attractions, and all of these are celebrated for good reason. But scratch beneath the surface and there’s far more to the city than 

In [21]:
raw_text = ""

# Combine the text content from all Document objects into a single string
# This creates one large corpus that can be used for text processing,
# analysis, or splitting when working with plain text instead of Document objects.
for i, doc in enumerate(docs):
    text = doc.page_content
    if text:
        raw_text += text

In [25]:
len(raw_text)

537183

In [26]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100,
    length_function = len,
    is_separator_regex = False,
)

In [27]:
texts = text_splitter.split_text(raw_text)

In [28]:
len(texts)

1567

In [29]:
print(texts[4])

If you still have energy to burn after ticking these must-dos off your list, check out the best things to do in Singapore this week and this weekend.

Quick picks: The best things to do in Singapore at a glance

Done something on this list and loved it? Share it with the hashtag #TimeOutSG.


## BGE Embeddings

In [30]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

In [31]:
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

In [32]:
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    #model_kwargs={'device': 'cuda'},
    encode_kwargs=encode_kwargs,
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\4097546878.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Vector DB

In [33]:
%%time
### Make the chroma and persiste to disk
db = Chroma.from_texts(texts,embedding_function,persist_directory="./chroma_db")

CPU times: total: 4min 11s
Wall time: 2min 33s


In [68]:
query = "Tell me about Universal Studios Singapore?"

results= db.similarity_search(query, k=5)

In [66]:
from pprint import pprint

for doc in results:
    pprint(doc.page_content)

('10. Visit Singaporeâs offshore islands\n'
 '\n'
 'If you want an escape from Singaporeâs ubiquitous skyscrapers, your best bet '
 'is to hop on a boat and sail to a nearby island for the day. Sentosa is the '
 'easiest to get to, and itâs home to a wide range of attractions, including '
 'white sandy beaches, Universal Studios and a casino.')
('Kampong Glam has emerged as one of the best places to visit in Singapore, '
 'largely thanks to its star attraction, Haji Lane. The bohemian street is as '
 'unbridled as it gets on the island, with energetic murals crawling up '
 'shophouses selling everything from clothing to trinkets. Make your way to '
 'the end at Beach Road (where the coastline used to be before reclamation '
 'happened) to experience a Mexican meal underneath a massive Aztec art piece '
 'at Piedra Niegra, or join the locals having some soupy')
('8. Take in Singaporeâs quirky side\n'
 '\n'
 'Beyond its gleaming towers and manicured parks, plenty of attractions will '
 '

## Setup a Retriever

In [50]:
retriever = db.as_retriever() # can add mmr fetch_k=20, search_type="mmr"

retriever.invoke(query)

[Document(id='c601d651-4168-4e7f-937f-019bf8830154', metadata={}, page_content='10. Visit Singaporeâs offshore islands\n\nIf you want an escape from Singaporeâs ubiquitous skyscrapers, your best bet is to hop on a boat and sail to a nearby island for the day. Sentosa is the easiest to get to, and itâs home to a wide range of attractions, including white sandy beaches, Universal Studios and a casino.'),
 Document(id='a9db7d49-be8e-4f4e-9c54-5568eee1909a', metadata={}, page_content='Kampong Glam has emerged as one of the best places to visit in Singapore, largely thanks to its star attraction, Haji Lane. The bohemian street is as unbridled as it gets on the island, with energetic murals crawling up shophouses selling everything from clothing to trinkets. Make your way to the end at Beach Road (where the coastline used to be before reclamation happened) to experience a Mexican meal underneath a massive Aztec art piece at Piedra Niegra, or join the locals having some soupy'),
 Document(id=

## Chat chain

In [52]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [53]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

In [56]:
prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [58]:
print(prompt.messages[0].prompt.template)

Answer the question based only on the following context:
{context}

Question: {question}



In [59]:
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', temperature=0.0, client=<google.genai.client.Client object at 0x000001E46F046AD0>, default_metadata=(), model_kwargs={})

In [60]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [61]:
text_reply = chain.invoke("Tell me about Universal Studio Singapore")

print(wrap_text(text_reply))

Universal Studios is an attraction located on Sentosa, one of Singapore's offshore
islands. Sentosa is also home to white sandy beaches and a casino.


In [64]:
text_reply = chain.invoke("What are 5 most beautiful rivers in Singapore?")

print(wrap_text(text_reply))

Based on the provided context, there is no information about rivers in Singapore,
beautiful or otherwise.


## With RagFusion

In [70]:
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatMessagePromptTemplate,
    PromptTemplate,
)

In [89]:
# Prompt to generate multiple search queries from a single user query (RAG Fusion)

prompt = ChatPromptTemplate(
    input_variables=["original_query"],
    messages=[
        # Define the LLM's role
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                template="You are a helpful assistant that generates multiple search queries from a single input query.",
            )
        ),

        # Ask the LLM to generate 4 related search queries
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=["original_query"],
                template=(
                    "Generate 4 search queries related to:\n"
                    "{question}\n\n"
                    "OUTPUT (4 queries):"
                ),
            )
        ),
    ],
)

In [90]:
original_query = "universal studios Singapore"

In [91]:
generate_queries = (
    prompt | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [92]:
generate_queries

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries from a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate 4 search queries related to:\n{question}\n\nOUTPUT (4 queries):'), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': T

In [93]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}

    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)

            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    return reranked_results

In [94]:
ragfusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

For example

User asks: Tell me about Universal Studios Singapore

After generate_queries

[
    "Universal Studios Singapore attractions",
    "Universal Studios Singapore history",
    "Universal Studios Singapore rides",
    "USS ticket prices",
    "Things to do in USS"
]

After retriever.map()

[
    [doc1, doc2, doc3],
    [doc4, doc2, doc1],
    [doc5, doc1, doc2],
    [doc2, doc6, doc7],
    [doc1, doc8, doc4]
]

After reciprocal_rank_fusion

[
    (doc1, 0.0489),
    (doc2, 0.0476),
    (doc4, 0.0312),
    (doc5, 0.0164),
    (doc6, 0.0159)
]

These are your best documents, ranked using information from all generated queries rather than just one.

What you get at the end

The variable ragfusion_chain is a runnable pipeline, not the results themselves. When you execute it, for example:

results = ragfusion_chain.invoke("Tell me about Universal Studios Singapore")

you receive a reranked list of (Document, score) tuples. These top-ranked Document objects are typically passed as the context to your LLM to generate the final answer in a RAG application.

In [96]:
ragfusion_chain.input_schema.schema()  # Returns the input schema of the LangChain Runnable.
# It tells you:
# - What inputs the chain expects
# - Input variable names
# - Input data types
# - Whether the inputs are required
#
# Think of it as asking:
# "What should I pass to this chain?"

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\116244941.py:1: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  ragfusion_chain.input_schema.schema()  # Returns the input schema of the LangChain Runnable.


{'properties': {'question': {'title': 'Question', 'type': 'string'}},
 'required': ['question'],
 'title': 'PromptInput',
 'type': 'object'}

In [99]:
ragfusion_chain.invoke({
    "question": "Tell me about Singapore's nightlife scene?"
})

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\3111552449.py:18: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc), score)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\3111552449.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


[(Document(id='71a9230e-27cf-4581-8ef5-6b9be6482d02', metadata={}, page_content='Here are my 10 favorite things to do in Singapore.\n\n1. Eat as much hawker food as you can'),
  0.04945355191256831),
 (Document(id='a9db7d49-be8e-4f4e-9c54-5568eee1909a', metadata={}, page_content='Kampong Glam has emerged as one of the best places to visit in Singapore, largely thanks to its star attraction, Haji Lane. The bohemian street is as unbridled as it gets on the island, with energetic murals crawling up shophouses selling everything from clothing to trinkets. Make your way to the end at Beach Road (where the coastline used to be before reclamation happened) to experience a Mexican meal underneath a massive Aztec art piece at Piedra Niegra, or join the locals having some soupy'),
  0.049189141547682),
 (Document(id='522f47b3-2f64-4cbe-96fa-5ee6c6bae6fd', metadata={}, page_content='RECOMMENDED:\n\nNew bars in Singapore: January 2026\n\nNew bars in Singapore: April 2026'),
  0.016666666666666666)

In [101]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

full_rag_fusion_chain = (
    {
        "context": ragfusion_chain,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [102]:
full_rag_fusion_chain.input_schema.schema()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\1333827808.py:1: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  full_rag_fusion_chain.input_schema.schema()


{'properties': {'question': {'title': 'Question', 'type': 'string'},
  'root': {'title': 'Root'}},
 'required': ['question', 'root'],
 'title': 'RunnableParallel<context,question>Input',
 'type': 'object'}

In [103]:
full_rag_fusion_chain.invoke({"question": "Tell me about Singapore’s nightlife scene?"})

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\3111552449.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


"Singapore's nightlife scene can be sampled along the Singapore River, where former warehouses have been converted into top nightlife zones. There is also a bar-centric area that is home to some of the world's best watering holes, including two Singaporean bars on the World's 50 Best list. New bars are also opening in Singapore.\n\nTo save on drinks, you can take advantage of supermarkets or hawker center drink stalls that serve bottled beer, but be aware there is a curfew on alcohol sales in Singapore after 10:30 pm. Alternatively, seek out happy hour deals in nightlife districts."

"Singapore's nightlife scene can be sampled along the Singapore River, where former warehouses have been converted into top nightlife zones. There is also a bar-centric area that is home to some of the world's best watering holes, including two Singaporean bars on the World's 50 Best list. New bars are also opening in Singapore.\n\nTo save on drinks, you can take advantage of supermarkets or hawker center drink stalls that serve bottled beer, but be aware there is a curfew on alcohol sales in Singapore after 10:30 pm. Alternatively, seek out happy hour deals in nightlife districts."

# Conclusion

In this notebook, I implemented **RAG Fusion**, an advanced retrieval technique that improves document retrieval by generating multiple search queries from a single user question. Instead of relying on a single query, the LLM generates several semantically different queries, enabling the retriever to discover more relevant information.

I then combined the retrieved documents from all generated queries using **Reciprocal Rank Fusion (RRF)**, which gives higher importance to documents that consistently appear across multiple retrieval results. This approach improves recall, reduces the impact of query wording, and produces a more reliable set of context documents.

Finally, I passed the reranked documents to the LLM to generate a context-aware answer. Compared to a standard single-query RAG pipeline, RAG Fusion retrieves broader and more relevant context, resulting in more accurate and comprehensive responses for complex or ambiguous questions.

Overall, this notebook demonstrates how RAG Fusion can significantly enhance the retrieval stage of a RAG pipeline while remaining simple to implement. It is an effective technique for building more robust, accurate, and reliable AI applications.
